# 2024 기업정보화통계조사 분포 확인 그림 생성

## 1. 목적

이 노트북은 2024년 기업정보화통계조사 최종 분석용 데이터셋을 기준으로, 9주차 중간보고서 4. 예비분석 결과 섹션에 넣을 분포 확인 그림을 생성한다.

원칙:
- 최종 분석용 데이터셋은 `working/analysis` 폴더에서만 읽는다.
- 원본 데이터 파일은 변경하지 않는다.
- 존재하지 않는 변수나 수치는 임의로 만들지 않는다.
- 각 그림은 보고서 삽입용 PNG로 `output/figure`에 저장한다.
- 모든 그림은 `matplotlib`만 사용하고 `dpi=300`으로 저장한다.

## 2. 데이터 로드 및 변수 확인

프로젝트 폴더 구조, 최종 분석용 데이터 후보, 변수 존재 여부를 확인한다. `ai_use_sum_group`은 Figure 4 생성용 임시 변수로만 사용하며 원본 데이터셋에는 저장하지 않는다.

In [1]:
from pathlib import Path
import platform
import math
import json
import os
import pandas as pd
import numpy as np

# macOS GUI backend 및 홈 디렉터리 폰트 캐시 권한 문제를 피하기 위한 설정
mpl_cache = Path('/private/tmp/matplotlib-codex-cache')
mpl_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(mpl_cache))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

# 노트북 실행 위치가 프로젝트 루트 또는 code 폴더일 수 있으므로 루트를 추정한다.
CWD = Path.cwd()
if (CWD / 'working').exists():
    BASE_DIR = CWD
elif (CWD.parent / 'working').exists():
    BASE_DIR = CWD.parent
else:
    raise FileNotFoundError('working 폴더가 있는 프로젝트 루트를 찾지 못했습니다.')

RAW_DIR = BASE_DIR / 'raw'
WORKING_DIR = BASE_DIR / 'working'
ANALYSIS_DIR = WORKING_DIR / 'analysis'
CODE_DIR = BASE_DIR / 'code'
FIG_DIR = BASE_DIR / 'output' / 'figure'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('BASE_DIR:', BASE_DIR)
print('FIG_DIR:', FIG_DIR)

required_dirs = [
    RAW_DIR, WORKING_DIR, CODE_DIR,
    WORKING_DIR / 'subset', WORKING_DIR / 'rename', WORKING_DIR / 'cleaned',
    WORKING_DIR / 'featured', WORKING_DIR / 'analysis'
]
folder_check = pd.DataFrame({
    'folder': [str(p.relative_to(BASE_DIR)) for p in required_dirs],
    'exists': [p.exists() and p.is_dir() for p in required_dirs],
})
print('\n[폴더 확인]')
print(folder_check.to_string(index=False))

BASE_DIR: /Users/yenarue/Downloads/ITM.89912(A)/연구 데이터
FIG_DIR: /Users/yenarue/Downloads/ITM.89912(A)/연구 데이터/output/figure

[폴더 확인]
          folder  exists
             raw    True
         working    True
            code    True
  working/subset    True
  working/rename    True
 working/cleaned    True
working/featured    True
working/analysis    True


In [2]:
# 최종 분석용 데이터셋 후보 확인: csv -> xlsx -> parquet 우선순위
patterns = ['*.csv', '*.xlsx', '*.parquet']
candidates = []
for priority, pattern in enumerate(patterns, start=1):
    for path in ANALYSIS_DIR.glob(pattern):
        candidates.append({
            'priority': priority,
            'file': path.name,
            'path': path,
            'modified_time': pd.Timestamp(path.stat().st_mtime, unit='s'),
            'size_bytes': path.stat().st_size,
        })

candidate_df = pd.DataFrame(candidates).sort_values(['priority', 'modified_time'], ascending=[True, False])
print('[working/analysis 데이터 후보]')
print(candidate_df[['priority', 'file', 'modified_time', 'size_bytes']].to_string(index=False))

if candidate_df.empty:
    raise FileNotFoundError('working/analysis 안에서 csv/xlsx/parquet 분석 파일을 찾지 못했습니다.')

DATA_PATH = candidate_df.iloc[0]['path']
print('\n사용 데이터 파일:', DATA_PATH.relative_to(BASE_DIR))

[working/analysis 데이터 후보]
 priority                        file                 modified_time  size_bytes
        1 nia_2024_analysis_total.csv 2026-04-27 19:01:15.774730444     1654412

사용 데이터 파일: working/analysis/nia_2024_analysis_total.csv


In [3]:
# 데이터 읽기: 최종 분석용 데이터셋만 사용한다.
suffix = DATA_PATH.suffix.lower()
if suffix == '.csv':
    df = pd.read_csv(DATA_PATH)
elif suffix in ['.xlsx', '.xls']:
    df = pd.read_excel(DATA_PATH)
elif suffix == '.parquet':
    df = pd.read_parquet(DATA_PATH)
else:
    raise ValueError(f'지원하지 않는 파일 형식입니다: {suffix}')

total_n = len(df)
print('전체 N:', f'{total_n:,}')
print('열 수:', df.shape[1])

required_vars = [
    'effect_proc_improve', 'effect_average', 'it_org_any', 'ai_use',
    'ai_use_sum', 'it_invest_sum', 'firm_size', 'industry'
]

variable_check = []
for var in required_vars:
    exists = var in df.columns
    variable_check.append({
        '변수명': var,
        '존재 여부': exists,
        '유효 N': int(df[var].notna().sum()) if exists else np.nan,
        '결측 N': int(df[var].isna().sum()) if exists else np.nan,
    })
variable_check = pd.DataFrame(variable_check)
missing_vars = variable_check.loc[~variable_check['존재 여부'], '변수명'].tolist()
print('\n[변수 존재 여부]')
print(variable_check.to_string(index=False))

if missing_vars:
    print('\n확인 필요 변수:', ', '.join(missing_vars))
else:
    print('\n요청 변수 8개가 모두 존재합니다.')

전체 N: 12,203
열 수: 58

[변수 존재 여부]
                변수명  존재 여부  유효 N  결측 N
effect_proc_improve   True 12203     0
     effect_average   True 12203     0
         it_org_any   True 12203     0
             ai_use   True 12203     0
         ai_use_sum   True 12203     0
      it_invest_sum   True 12203     0
          firm_size   True 12203     0
           industry   True 12203     0

요청 변수 8개가 모두 존재합니다.


In [4]:
# matplotlib 한글 폰트 설정: Mac AppleGothic, Windows Malgun Gothic, Linux/Colab NanumGothic 우선 시도
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
font_candidates = ['AppleGothic', 'Malgun Gothic', 'NanumGothic']
selected_font = next((font for font in font_candidates if font in available_fonts), None)
USE_KOREAN = selected_font is not None

if USE_KOREAN:
    rcParams['font.family'] = selected_font
else:
    rcParams['font.family'] = 'DejaVu Sans'

rcParams['axes.unicode_minus'] = False
rcParams['figure.dpi'] = 120

print('사용 폰트:', selected_font if selected_font else 'DejaVu Sans (English fallback)')
print('한글 라벨 사용:', USE_KOREAN)

사용 폰트: AppleGothic
한글 라벨 사용: True


In [5]:
# 코드북에서 firm_size, industry 라벨 확인
CODEBOOK_PATH = RAW_DIR / 'nia_2024_codebook.csv'

def labels_from_codebook(path: Path, raw_var: str) -> dict:
    if not path.exists():
        return {}
    cb = pd.read_csv(path, encoding='utf-8-sig')
    var_col, val_col, lab_col = cb.columns[0], cb.columns[2], cb.columns[3]
    starts = cb.index[cb[var_col].astype(str).str.strip().eq(raw_var)].tolist()
    if not starts:
        return {}
    start = starts[0]
    labels = {}
    for i in range(start, len(cb)):
        first = cb.loc[i, var_col]
        if i > start and pd.notna(first) and str(first).strip():
            break
        value = cb.loc[i, val_col]
        label = cb.loc[i, lab_col]
        if pd.notna(value) and pd.notna(label):
            key = int(value) if float(value).is_integer() else float(value)
            labels[key] = str(label).strip()
    return labels

firm_size_labels = labels_from_codebook(CODEBOOK_PATH, 'D_SIZE')
industry_labels = labels_from_codebook(CODEBOOK_PATH, 'D_IND')

if not firm_size_labels:
    firm_size_labels = {1: '10-49명', 2: '50-249명', 3: '250~999명', 4: '1000명 이상'}

print('firm_size 라벨:', firm_size_labels)
print('industry 라벨 확인:', bool(industry_labels))

firm_size 라벨: {1: '10-49명', 2: '50-249명', 3: '250~999명', 4: '1000명 이상'}
industry 라벨 확인: True


## 3. 본문용 핵심 그림

본문용 Figure 1-4를 생성한다. 결측값은 각 그림별로 필요한 변수 기준으로 pairwise 제외하고, 제외된 N을 해석 메모에 기록한다.

In [6]:
# 공통 함수
saved_figures = []
saved_tables = []
interpretations = []
check_notes = []

if missing_vars:
    check_notes.append('최종 분석용 데이터셋에 없는 변수: ' + ', '.join(missing_vars))

K = {
    'count': '기업 수',
    'percent': '비율(%)',
    'effect_proc': '프로세스 개선 효과',
    'ai_sum': 'AI 활용 유형 개수',
    'ai_sum_mean': 'AI 활용 유형 개수 평균',
    'it_org': '정보화 담당 체계 보유 여부',
    'no_org': '미보유',
    'yes_org': '보유',
    'firm_size': '기업 규모',
    'industry': '업종',
    'mean': '평균',
}
E = {
    'count': 'Number of firms',
    'percent': 'Percent (%)',
    'effect_proc': 'Process improvement effect',
    'ai_sum': 'Number of AI use types',
    'ai_sum_mean': 'Mean number of AI use types',
    'it_org': 'IT organization status',
    'no_org': 'No IT org',
    'yes_org': 'Has IT org',
    'firm_size': 'Firm size',
    'industry': 'Industry',
    'mean': 'Mean',
}
L = K if USE_KOREAN else E

def save_fig(fig, filename):
    path = FIG_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    saved_figures.append(path)
    return path

def ci95(series):
    s = pd.to_numeric(series, errors='coerce').dropna()
    n = len(s)
    mean = float(s.mean()) if n else np.nan
    sd = float(s.std(ddof=1)) if n > 1 else np.nan
    se = sd / math.sqrt(n) if n > 1 else np.nan
    ci = 1.96 * se if n > 1 else np.nan
    return mean, sd, se, mean - ci if n > 1 else np.nan, mean + ci if n > 1 else np.nan, n

def pct_label(value):
    return f'{value:.1f}%'

In [7]:
# Figure 1. 프로세스 개선 효과 분포
vars_needed = ['effect_proc_improve']
fig1_df = df[vars_needed].dropna().copy()
fig1_excluded = total_n - len(fig1_df)
counts = fig1_df['effect_proc_improve'].value_counts().sort_index()
fig1_data = pd.DataFrame({
    'effect_proc_improve': counts.index.astype(int),
    'N': counts.values.astype(int),
})
fig1_data['비율(%)'] = fig1_data['N'] / len(fig1_df) * 100
fig1_data['비율(%)'] = fig1_data['비율(%)'].round(3)
fig1_data_path = FIG_DIR / 'fig1_effect_proc_improve_distribution_data.csv'
fig1_data.to_csv(fig1_data_path, index=False, encoding='utf-8-sig')
saved_tables.append(fig1_data_path)

fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.bar(fig1_data['effect_proc_improve'], fig1_data['비율(%)'], color='#4C78A8', edgecolor='white')
ax.set_title('Figure 1. 프로세스 개선 효과 분포' if USE_KOREAN else 'Figure 1. Distribution of Process Improvement Effect')
ax.set_xlabel(L['effect_proc'])
ax.set_ylabel(L['percent'])
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_ylim(0, max(fig1_data['비율(%)']) * 1.18)
for _, row in fig1_data.iterrows():
    ax.text(row['effect_proc_improve'], row['비율(%)'] + 0.7, pct_label(row['비율(%)']), ha='center', va='bottom', fontsize=9)
ax.grid(axis='y', alpha=0.25)
fig1_path = save_fig(fig, 'fig1_effect_proc_improve_distribution.png')

mode_row = fig1_data.loc[fig1_data['비율(%)'].idxmax()]
high_pct = fig1_data.loc[fig1_data['effect_proc_improve'].isin([4, 5]), '비율(%)'].sum()
interpretations.append({
    '그림': 'Figure 1',
    '파일': fig1_path.name,
    '해석': f'프로세스 개선 효과는 {int(mode_row["effect_proc_improve"])}점 응답이 {mode_row["비율(%)"]:.1f}%로 가장 많고, 4점 이상 응답은 {high_pct:.1f}%로 전반적으로 높은 값에 집중되어 있다. 그림 생성 시 제외된 결측은 {fig1_excluded:,}건이다.'
})
print(fig1_path.relative_to(BASE_DIR))

output/figure/fig1_effect_proc_improve_distribution.png


In [8]:
# Figure 2. 정보화 담당 체계 보유 여부에 따른 프로세스 개선 효과
vars_needed = ['it_org_any', 'effect_proc_improve']
fig2_df = df[vars_needed].dropna().copy()
fig2_excluded = total_n - len(fig2_df)
fig2_df['it_org_any'] = fig2_df['it_org_any'].astype(int)

group_rows = []
for value, sub in fig2_df.groupby('it_org_any'):
    mean, sd, se, lo, hi, n = ci95(sub['effect_proc_improve'])
    group_rows.append({'it_org_any': value, 'label': '정보화 담당 체계 미보유' if value == 0 else '정보화 담당 체계 보유', 'N': n, 'mean': mean, 'sd': sd, 'se': se, 'ci95_low': lo, 'ci95_high': hi})
fig2_data = pd.DataFrame(group_rows).sort_values('it_org_any')
fig2_data[['mean', 'sd', 'se', 'ci95_low', 'ci95_high']] = fig2_data[['mean', 'sd', 'se', 'ci95_low', 'ci95_high']].round(3)
fig2_data_path = FIG_DIR / 'fig2_effect_by_it_org_data.csv'
fig2_data.to_csv(fig2_data_path, index=False, encoding='utf-8-sig')
saved_tables.append(fig2_data_path)

labels = [L['no_org'], L['yes_org']]
data = [fig2_df.loc[fig2_df['it_org_any'] == 0, 'effect_proc_improve'], fig2_df.loc[fig2_df['it_org_any'] == 1, 'effect_proc_improve']]
fig, ax = plt.subplots(figsize=(7.2, 4.8))
box = ax.boxplot(data, tick_labels=labels, patch_artist=True, showfliers=False, widths=0.55)
for patch, color in zip(box['boxes'], ['#F58518', '#4C78A8']):
    patch.set_facecolor(color)
    patch.set_alpha(0.72)
means = fig2_data['mean'].values
ax.scatter([1, 2], means, color='black', marker='D', s=42, zorder=3, label=L['mean'])
for x, mean in zip([1, 2], means):
    ax.text(x, mean + 0.08, f'{mean:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_title('Figure 2. 정보화 담당 체계 보유 여부에 따른 프로세스 개선 효과' if USE_KOREAN else 'Figure 2. Process Improvement by IT Organization Status')
ax.set_xlabel(L['it_org'])
ax.set_ylabel(L['effect_proc'])
ax.set_ylim(1, 5.25)
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.25)
fig2_path = save_fig(fig, 'fig2_effect_by_it_org_boxplot.png')

diff = fig2_data.loc[fig2_data['it_org_any'] == 1, 'mean'].iloc[0] - fig2_data.loc[fig2_data['it_org_any'] == 0, 'mean'].iloc[0]
interpretations.append({
    '그림': 'Figure 2',
    '파일': fig2_path.name,
    '해석': f'정보화 담당 체계 보유 기업의 프로세스 개선 효과 평균은 {fig2_data.loc[fig2_data["it_org_any"] == 1, "mean"].iloc[0]:.3f}점, 미보유 기업은 {fig2_data.loc[fig2_data["it_org_any"] == 0, "mean"].iloc[0]:.3f}점으로 보유 기업이 {diff:.3f}점 높다. 그림 생성 시 제외된 결측은 {fig2_excluded:,}건이다.'
})
print(fig2_path.relative_to(BASE_DIR))

output/figure/fig2_effect_by_it_org_boxplot.png


In [9]:
# Figure 3. 정보화 담당 체계 보유 여부에 따른 AI 활용 수준
vars_needed = ['it_org_any', 'ai_use_sum']
fig3_df = df[vars_needed].dropna().copy()
fig3_excluded = total_n - len(fig3_df)
fig3_df['it_org_any'] = fig3_df['it_org_any'].astype(int)

rows = []
for value, sub in fig3_df.groupby('it_org_any'):
    mean, sd, se, lo, hi, n = ci95(sub['ai_use_sum'])
    rows.append({'it_org_any': value, 'label': '정보화 담당 체계 미보유' if value == 0 else '정보화 담당 체계 보유', 'N': n, 'mean': mean, 'sd': sd, 'se': se, 'ci95_low': lo, 'ci95_high': hi})
fig3_data = pd.DataFrame(rows).sort_values('it_org_any')
fig3_data[['mean', 'sd', 'se', 'ci95_low', 'ci95_high']] = fig3_data[['mean', 'sd', 'se', 'ci95_low', 'ci95_high']].round(3)
fig3_data_path = FIG_DIR / 'fig3_ai_use_sum_by_it_org_data.csv'
fig3_data.to_csv(fig3_data_path, index=False, encoding='utf-8-sig')
saved_tables.append(fig3_data_path)

x = np.arange(len(fig3_data))
y = fig3_data['mean'].values
yerr = 1.96 * fig3_data['se'].values
fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.bar(x, y, yerr=yerr, capsize=6, color=['#F58518', '#4C78A8'], edgecolor='white', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([L['no_org'], L['yes_org']])
ax.set_title('Figure 3. 정보화 담당 체계 보유 여부에 따른 AI 활용 수준' if USE_KOREAN else 'Figure 3. AI Use Intensity by IT Organization Status')
ax.set_xlabel(L['it_org'])
ax.set_ylabel(L['ai_sum_mean'])
ax.set_ylim(0, max(y + yerr) * 1.25)
for xi, yi in zip(x, y):
    ax.text(xi, yi + 0.06, f'{yi:.3f}', ha='center', va='bottom', fontsize=9)
ax.grid(axis='y', alpha=0.25)
fig3_path = save_fig(fig, 'fig3_ai_use_sum_by_it_org.png')

diff = fig3_data.loc[fig3_data['it_org_any'] == 1, 'mean'].iloc[0] - fig3_data.loc[fig3_data['it_org_any'] == 0, 'mean'].iloc[0]
interpretations.append({
    '그림': 'Figure 3',
    '파일': fig3_path.name,
    '해석': f'정보화 담당 체계 보유 기업의 AI 활용 유형 개수 평균은 {fig3_data.loc[fig3_data["it_org_any"] == 1, "mean"].iloc[0]:.3f}개, 미보유 기업은 {fig3_data.loc[fig3_data["it_org_any"] == 0, "mean"].iloc[0]:.3f}개로 보유 기업이 {diff:.3f}개 높다. 그림 생성 시 제외된 결측은 {fig3_excluded:,}건이다.'
})
print(fig3_path.relative_to(BASE_DIR))

output/figure/fig3_ai_use_sum_by_it_org.png


In [10]:
# Figure 4. AI 활용 수준에 따른 정보화 담당 체계와 프로세스 개선 효과의 관계
vars_needed = ['effect_proc_improve', 'it_org_any', 'ai_use_sum']
fig4_df = df[vars_needed].dropna().copy()
fig4_excluded = total_n - len(fig4_df)
fig4_df['it_org_any'] = fig4_df['it_org_any'].astype(int)
fig4_df['ai_use_sum_group'] = np.select(
    [fig4_df['ai_use_sum'].eq(0), fig4_df['ai_use_sum'].eq(1), fig4_df['ai_use_sum'].ge(2)],
    [0, 1, 2],
    default=np.nan
).astype(int)

group_labels_kr = {0: '0개/미활용', 1: '1개', 2: '2개 이상'}
group_labels_en = {0: '0 types', 1: '1 type', 2: '2+ types'}
group_labels = group_labels_kr if USE_KOREAN else group_labels_en

rows = []
for (ai_group, org), sub in fig4_df.groupby(['ai_use_sum_group', 'it_org_any']):
    mean, sd, se, lo, hi, n = ci95(sub['effect_proc_improve'])
    rows.append({'ai_use_sum_group': ai_group, 'ai_use_sum_group_label': group_labels_kr[ai_group], 'it_org_any': org, 'it_org_label': '정보화 담당 체계 미보유' if org == 0 else '정보화 담당 체계 보유', 'N': n, 'mean': mean, 'sd': sd, 'se': se, 'ci95_low': lo, 'ci95_high': hi})
fig4_data = pd.DataFrame(rows).sort_values(['ai_use_sum_group', 'it_org_any'])
fig4_data[['mean', 'sd', 'se', 'ci95_low', 'ci95_high']] = fig4_data[['mean', 'sd', 'se', 'ci95_low', 'ci95_high']].round(3)
fig4_data_path = FIG_DIR / 'fig4_interaction_data.csv'
fig4_data.to_csv(fig4_data_path, index=False, encoding='utf-8-sig')
saved_tables.append(fig4_data_path)

fig, ax = plt.subplots(figsize=(7.8, 4.9))
x = np.arange(3)
width = 0.34
colors = {0: '#F58518', 1: '#4C78A8'}
for org, offset in [(0, -width/2), (1, width/2)]:
    sub = fig4_data[fig4_data['it_org_any'] == org].set_index('ai_use_sum_group').reindex([0, 1, 2])
    means = sub['mean'].values
    errs = 1.96 * sub['se'].values
    bars = ax.bar(x + offset, means, width, yerr=errs, capsize=5, color=colors[org], edgecolor='white', alpha=0.85, label=L['no_org'] if org == 0 else L['yes_org'])
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, mean + 0.05, f'{mean:.3f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels([group_labels[i] for i in [0, 1, 2]])
ax.set_title('Figure 4. AI 활용 수준별 정보화 담당 체계와 프로세스 개선 효과' if USE_KOREAN else 'Figure 4. IT Organization and Process Improvement by AI Use Level')
ax.set_xlabel(L['ai_sum'])
ax.set_ylabel(L['effect_proc'])
ax.set_ylim(1, 5.25)
ax.legend(title=L['it_org'], frameon=False)
ax.grid(axis='y', alpha=0.25)
fig4_path = save_fig(fig, 'fig4_interaction_it_org_ai_use_effect.png')

low_diff = fig4_data.query('ai_use_sum_group == 0 and it_org_any == 1')['mean'].iloc[0] - fig4_data.query('ai_use_sum_group == 0 and it_org_any == 0')['mean'].iloc[0]
high_diff = fig4_data.query('ai_use_sum_group == 2 and it_org_any == 1')['mean'].iloc[0] - fig4_data.query('ai_use_sum_group == 2 and it_org_any == 0')['mean'].iloc[0]
interpretations.append({
    '그림': 'Figure 4',
    '파일': fig4_path.name,
    '해석': f'AI 활용 유형 0개 집단에서 정보화 담당 체계 보유-미보유 평균 차이는 {low_diff:.3f}점이고, 2개 이상 집단에서는 {high_diff:.3f}점으로 나타났다. 이는 AI 활용 수준별로 정보화 담당 체계와 프로세스 개선 효과의 관계가 달라지는지 시각적으로 점검하기 위한 결과이며, 제외된 결측은 {fig4_excluded:,}건이다.'
})
print(fig4_path.relative_to(BASE_DIR))

output/figure/fig4_interaction_it_org_ai_use_effect.png


## 4. 부록용 보조 그림

부록용 Figure A1-A4를 생성한다. 카운트형 변수의 0 집중, 투자 항목 분포, 기업 규모와 업종 표본 구성을 확인한다.

In [11]:
# Appendix distribution data를 누적 저장하기 위한 리스트
appendix_rows = []

In [12]:
# Figure A1. AI 활용 유형 개수 분포
vars_needed = ['ai_use_sum']
a1_df = df[vars_needed].dropna().copy()
a1_excluded = total_n - len(a1_df)
a1_counts = a1_df['ai_use_sum'].value_counts().sort_index()
a1_data = pd.DataFrame({'variable': 'ai_use_sum', 'value': a1_counts.index.astype(int), 'label': a1_counts.index.astype(int).astype(str), 'N': a1_counts.values.astype(int)})
a1_data['비율(%)'] = (a1_data['N'] / len(a1_df) * 100).round(3)
appendix_rows.append(a1_data)

ai_mean = float(a1_df['ai_use_sum'].mean())
ai_var = float(a1_df['ai_use_sum'].var(ddof=1))
ai_zero_pct = float((a1_df['ai_use_sum'] == 0).sum() / len(a1_df) * 100)
ai_vmr = ai_var / ai_mean

fig, ax = plt.subplots(figsize=(7.4, 4.7))
ax.bar(a1_data['value'], a1_data['비율(%)'], color='#72B7B2', edgecolor='white')
ax.set_title('Figure A1. AI 활용 유형 개수 분포' if USE_KOREAN else 'Figure A1. Distribution of AI Use Types')
ax.set_xlabel(L['ai_sum'])
ax.set_ylabel(L['percent'])
ax.set_xticks(a1_data['value'])
ax.set_ylim(0, max(a1_data['비율(%)']) * 1.20)
for _, row in a1_data.iterrows():
    ax.text(row['value'], row['비율(%)'] + 0.7, pct_label(row['비율(%)']), ha='center', va='bottom', fontsize=8)
ax.text(0.98, 0.95, f'Mean={ai_mean:.3f}\nVar/Mean={ai_vmr:.3f}\nZero={ai_zero_pct:.1f}%', transform=ax.transAxes, ha='right', va='top', fontsize=9, bbox={'boxstyle': 'round,pad=0.3', 'facecolor': 'white', 'edgecolor': '#cccccc'})
ax.grid(axis='y', alpha=0.25)
a1_path = save_fig(fig, 'figA1_ai_use_sum_distribution.png')
interpretations.append({
    '그림': 'Figure A1',
    '파일': a1_path.name,
    '해석': f'AI 활용 유형 개수는 0개가 {ai_zero_pct:.1f}%로 가장 많고 평균은 {ai_mean:.3f}개이다. 분산/평균 비율은 {ai_vmr:.3f}로 1보다 커 카운트형 대안모형 검토가 필요하며, 제외된 결측은 {a1_excluded:,}건이다.'
})
print(a1_path.relative_to(BASE_DIR))

output/figure/figA1_ai_use_sum_distribution.png


In [13]:
# Figure A2. 정보화 투자 항목 개수 분포
vars_needed = ['it_invest_sum']
a2_df = df[vars_needed].dropna().copy()
a2_excluded = total_n - len(a2_df)
a2_counts = a2_df['it_invest_sum'].value_counts().sort_index()
a2_data = pd.DataFrame({'variable': 'it_invest_sum', 'value': a2_counts.index.astype(int), 'label': a2_counts.index.astype(int).astype(str), 'N': a2_counts.values.astype(int)})
a2_data['비율(%)'] = (a2_data['N'] / len(a2_df) * 100).round(3)
appendix_rows.append(a2_data)

fig, ax = plt.subplots(figsize=(7.4, 4.7))
ax.bar(a2_data['value'], a2_data['비율(%)'], color='#54A24B', edgecolor='white')
ax.set_title('Figure A2. 정보화 투자 항목 개수 분포' if USE_KOREAN else 'Figure A2. Distribution of IT Investment Items')
ax.set_xlabel('정보화 투자 항목 개수' if USE_KOREAN else 'Number of IT investment items')
ax.set_ylabel(L['percent'])
ax.set_xticks(a2_data['value'])
ax.set_ylim(0, max(a2_data['비율(%)']) * 1.20)
for _, row in a2_data.iterrows():
    ax.text(row['value'], row['비율(%)'] + 0.5, pct_label(row['비율(%)']), ha='center', va='bottom', fontsize=8)
ax.grid(axis='y', alpha=0.25)
a2_path = save_fig(fig, 'figA2_it_invest_sum_distribution.png')
mode_row = a2_data.loc[a2_data['비율(%)'].idxmax()]
interpretations.append({
    '그림': 'Figure A2',
    '파일': a2_path.name,
    '해석': f'정보화 투자 항목 개수는 {int(mode_row["value"])}개가 {mode_row["비율(%)"]:.1f}%로 가장 높은 비중을 보이며, 평균은 {a2_df["it_invest_sum"].mean():.3f}개이다. 그림 생성 시 제외된 결측은 {a2_excluded:,}건이다.'
})
print(a2_path.relative_to(BASE_DIR))

output/figure/figA2_it_invest_sum_distribution.png


In [14]:
# Figure A3. 기업 규모 분포
vars_needed = ['firm_size']
a3_df = df[vars_needed].dropna().copy()
a3_excluded = total_n - len(a3_df)
a3_counts = a3_df['firm_size'].value_counts().sort_index()
a3_data = pd.DataFrame({'variable': 'firm_size', 'value': a3_counts.index.astype(int), 'label': [firm_size_labels.get(int(v), '라벨 확인 필요') for v in a3_counts.index], 'N': a3_counts.values.astype(int)})
a3_data['비율(%)'] = (a3_data['N'] / len(a3_df) * 100).round(3)
appendix_rows.append(a3_data)

fig, ax = plt.subplots(figsize=(7.5, 4.8))
x = np.arange(len(a3_data))
ax.bar(x, a3_data['비율(%)'], color='#B279A2', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(a3_data['label'], rotation=0)
ax.set_title('Figure A3. 기업 규모 분포' if USE_KOREAN else 'Figure A3. Firm Size Distribution')
ax.set_xlabel(L['firm_size'])
ax.set_ylabel(L['percent'])
ax.set_ylim(0, max(a3_data['비율(%)']) * 1.18)
for xi, pct in zip(x, a3_data['비율(%)']):
    ax.text(xi, pct + 0.7, pct_label(pct), ha='center', va='bottom', fontsize=9)
ax.grid(axis='y', alpha=0.25)
a3_path = save_fig(fig, 'figA3_firm_size_distribution.png')
small_medium_pct = a3_data.loc[a3_data['value'].isin([1, 2]), '비율(%)'].sum()
interpretations.append({
    '그림': 'Figure A3',
    '파일': a3_path.name,
    '해석': f'기업 규모는 10-49명 기업이 {a3_data.loc[a3_data["value"] == 1, "비율(%)"].iloc[0]:.1f}%로 가장 많고, 249명 이하 기업이 전체의 {small_medium_pct:.1f}%를 차지한다. 그림 생성 시 제외된 결측은 {a3_excluded:,}건이다.'
})
print(a3_path.relative_to(BASE_DIR))

output/figure/figA3_firm_size_distribution.png


In [15]:
# Figure A4. 업종 분포
vars_needed = ['industry']
a4_df = df[vars_needed].dropna().copy()
a4_excluded = total_n - len(a4_df)
a4_counts = a4_df['industry'].value_counts().sort_index()
if industry_labels:
    labels = [industry_labels.get(int(v), '라벨 확인 필요') for v in a4_counts.index]
else:
    labels = [str(int(v)) for v in a4_counts.index]
    check_notes.append('industry 라벨 확인 필요: 코드북에서 업종 라벨을 찾지 못해 숫자 코드로 표시함')
a4_data = pd.DataFrame({'variable': 'industry', 'value': a4_counts.index.astype(int), 'label': labels, 'N': a4_counts.values.astype(int)})
a4_data['비율(%)'] = (a4_data['N'] / len(a4_df) * 100).round(3)
appendix_rows.append(a4_data)

plot_data = a4_data.sort_values('비율(%)', ascending=True)
fig_height = max(5.5, 0.34 * len(plot_data))
fig, ax = plt.subplots(figsize=(8.5, fig_height))
y = np.arange(len(plot_data))
ax.barh(y, plot_data['비율(%)'], color='#E45756', edgecolor='white', alpha=0.85)
ax.set_yticks(y)
ax.set_yticklabels(plot_data['label'])
ax.set_title('Figure A4. 업종 분포' if USE_KOREAN else 'Figure A4. Industry Distribution')
ax.set_xlabel(L['percent'])
ax.set_ylabel(L['industry'])
ax.set_xlim(0, max(plot_data['비율(%)']) * 1.22)
for yi, pct in zip(y, plot_data['비율(%)']):
    ax.text(pct + 0.25, yi, pct_label(pct), va='center', fontsize=8)
ax.grid(axis='x', alpha=0.25)
a4_path = save_fig(fig, 'figA4_industry_distribution.png')
top3 = a4_data.sort_values('비율(%)', ascending=False).head(3)
top3_pct = top3['비율(%)'].sum()
interpretations.append({
    '그림': 'Figure A4',
    '파일': a4_path.name,
    '해석': f'업종별로는 {top3.iloc[0]["label"]}이 {top3.iloc[0]["비율(%)"]:.1f}%로 가장 많고, 상위 3개 업종의 합계 비중은 {top3_pct:.1f}%이다. 그림 생성 시 제외된 결측은 {a4_excluded:,}건이다.'
})
print(a4_path.relative_to(BASE_DIR))

output/figure/figA4_industry_distribution.png


In [16]:
# Appendix 보조 분포 데이터 저장
appendix_distribution_data = pd.concat(appendix_rows, ignore_index=True)
appendix_path = FIG_DIR / 'appendix_distribution_data.csv'
appendix_distribution_data.to_csv(appendix_path, index=False, encoding='utf-8-sig')
saved_tables.append(appendix_path)
print(appendix_path.relative_to(BASE_DIR))

output/figure/appendix_distribution_data.csv


## 5. 그림별 해석 메모

각 그림별로 보고서에 붙일 수 있는 1-2문장 해석을 작성하고, `output/figure/figure_interpretation.md`에 저장한다.

In [17]:
interpretation_df = pd.DataFrame(interpretations)
print(interpretation_df.to_string(index=False))

md_lines = []
md_lines.append('# 분포 확인 그림 해석 메모')
md_lines.append('')
md_lines.append(f'- 사용 데이터 파일: `{DATA_PATH.relative_to(BASE_DIR)}`')
md_lines.append(f'- 전체 N: {total_n:,}')
md_lines.append('')
for item in interpretations:
    md_lines.append(f'## {item["그림"]}. {item["파일"]}')
    md_lines.append(f'- {item["해석"]}')
    md_lines.append('')

if check_notes:
    md_lines.append('## 확인 필요 사항')
    for note in check_notes:
        md_lines.append(f'- {note}')
else:
    md_lines.append('## 확인 필요 사항')
    md_lines.append('- 요청 변수 8개가 모두 존재하며, 코드북에서 firm_size와 industry 라벨을 확인했다.')

interpretation_path = FIG_DIR / 'figure_interpretation.md'
interpretation_path.write_text('\n'.join(md_lines) + '\n', encoding='utf-8')
print('\n저장:', interpretation_path.relative_to(BASE_DIR))

       그림                                        파일                                                                                                                                                             해석
 Figure 1 fig1_effect_proc_improve_distribution.png                                                                   프로세스 개선 효과는 4점 응답이 49.5%로 가장 많고, 4점 이상 응답은 74.7%로 전반적으로 높은 값에 집중되어 있다. 그림 생성 시 제외된 결측은 0건이다.
 Figure 2         fig2_effect_by_it_org_boxplot.png                                                               정보화 담당 체계 보유 기업의 프로세스 개선 효과 평균은 4.100점, 미보유 기업은 3.838점으로 보유 기업이 0.262점 높다. 그림 생성 시 제외된 결측은 0건이다.
 Figure 3             fig3_ai_use_sum_by_it_org.png                                                               정보화 담당 체계 보유 기업의 AI 활용 유형 개수 평균은 1.022개, 미보유 기업은 0.720개로 보유 기업이 0.302개 높다. 그림 생성 시 제외된 결측은 0건이다.
 Figure 4 fig4_interaction_it_org_ai_use_effect.png AI 활용 유형 0개 집단에서 정보화 담당 체계 보유-미보유 평균 차이는 0.130점이고, 2개 이상 집단에서는 0.397점으로 나타났다. 이는 AI 활용 수준별로 정보화 담당 체계와 프

## 6. 확인 필요 사항

최종 출력으로 사용 데이터 파일명, 전체 N, 저장된 그림 파일, 저장된 보조 데이터 파일, 그림별 해석 메모, 확인 필요 사항을 점검한다.

In [18]:
print('1. 사용한 데이터 파일명:', DATA_PATH.relative_to(BASE_DIR))
print('2. 전체 N:', f'{total_n:,}')
print('\n3. 저장된 그림 파일 목록')
for path in saved_figures:
    print('-', path.relative_to(BASE_DIR))
print('\n4. 저장된 보조 데이터 파일 목록')
for path in saved_tables:
    print('-', path.relative_to(BASE_DIR))
print('\n5. 그림별 해석 메모')
for item in interpretations:
    print(f'- {item["그림"]}: {item["해석"]}')
print('\n6. 확인 필요 사항')
if check_notes:
    for note in check_notes:
        print('-', note)
else:
    print('- 요청 변수 8개가 모두 존재하며, 코드북에서 firm_size와 industry 라벨을 확인했다.')

1. 사용한 데이터 파일명: working/analysis/nia_2024_analysis_total.csv
2. 전체 N: 12,203

3. 저장된 그림 파일 목록
- output/figure/fig1_effect_proc_improve_distribution.png
- output/figure/fig2_effect_by_it_org_boxplot.png
- output/figure/fig3_ai_use_sum_by_it_org.png
- output/figure/fig4_interaction_it_org_ai_use_effect.png
- output/figure/figA1_ai_use_sum_distribution.png
- output/figure/figA2_it_invest_sum_distribution.png
- output/figure/figA3_firm_size_distribution.png
- output/figure/figA4_industry_distribution.png

4. 저장된 보조 데이터 파일 목록
- output/figure/fig1_effect_proc_improve_distribution_data.csv
- output/figure/fig2_effect_by_it_org_data.csv
- output/figure/fig3_ai_use_sum_by_it_org_data.csv
- output/figure/fig4_interaction_data.csv
- output/figure/appendix_distribution_data.csv

5. 그림별 해석 메모
- Figure 1: 프로세스 개선 효과는 4점 응답이 49.5%로 가장 많고, 4점 이상 응답은 74.7%로 전반적으로 높은 값에 집중되어 있다. 그림 생성 시 제외된 결측은 0건이다.
- Figure 2: 정보화 담당 체계 보유 기업의 프로세스 개선 효과 평균은 4.100점, 미보유 기업은 3.838점으로 보유 기업이 0.262점 높다. 그림 생성 시 제외된 결측은 0